# 01 · Data Collection & Initial Description
**CRISP-DM Phase 2 — Data Understanding: *Collect initial data* and *Describe data***

**Goal:** document where the data comes from, how it was collected, and what each field means.

| Decision | Value |
|---|---|
| Source | Binance spot REST API `GET /api/v3/klines` via `python-binance` (no API key needed) |
| Symbol | `BTCUSDT` (Bitcoin vs Tether, the most liquid BTC pair on Binance) |
| Interval | `1h` candles (resampled later to `4h` and `1D`) |
| Range | 2017-08-17 (Binance listing) → **2026-09-30 23:00 UTC (frozen cutoff)** |

The snapshot was produced once with `python -m src.data.collect` and committed to `data/raw/`.
It is **immutable**: every later step regenerates its outputs from this file, so all five team members work on identical data.

In [7]:
import sys
from pathlib import Path

# Make `src` importable whether the notebook runs from notebooks/ or the project root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

In [8]:
import json

import pandas as pd

from src import config
from src.data.collect import load_raw, snapshot_path

raw_path = snapshot_path(config.SYMBOL, config.BASE_INTERVAL, config.START_DATE, config.CUTOFF_DATE)
metadata = json.loads(raw_path.with_name(raw_path.stem + "_metadata.json").read_text())
metadata

{'symbol': 'BTCUSDT',
 'interval': '1h',
 'requested_start': '2017-08-17T00:00:00+00:00',
 'requested_end': '2026-09-30T23:00:00+00:00',
 'first_open_time': '2017-08-17T04:00:00+00:00',
 'last_open_time': '2026-09-30T23:00:00+00:00',
 'row_count': 79836,
 'collected_at': '2026-10-06T21:37:23+00:00',
 'source': 'Binance REST /api/v3/klines via python-binance',
 'python_binance_version': '1.0.37'}

The metadata file is our **provenance record**: what was requested, what came back, when, and with which library version.

Note that the first candle is at `04:00 UTC`, not `00:00`. Binance opened BTCUSDT trading a few hours into 17 August 2017.

In [9]:
raw = load_raw()
print(f"Shape: {raw.shape[0]:,} rows × {raw.shape[1]} columns")
print(f"Span:  {raw.index.min()} → {raw.index.max()}  ({(raw.index.max() - raw.index.min()).days:,} days)")
raw.dtypes

Shape: 79,836 rows × 10 columns
Span:  2017-08-17 04:00:00+00:00 → 2026-09-30 23:00:00+00:00  (3,331 days)


open                                  float64
high                                  float64
low                                   float64
close                                 float64
volume                                float64
close_time                datetime64[ms, UTC]
quote_asset_volume                    float64
number_of_trades                        int64
taker_buy_base_volume                 float64
taker_buy_quote_volume                float64
dtype: object

In [10]:
raw.head()

,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume
open_time,,,,,,,,,,
2017-08-17 04:00:00+00:00,4261.48,4313.62,4261.32,4308.83,47.181009,2017-08-17 04:59:59.999000+00:00,202366.138393,171,35.160503,150952.477943
2017-08-17 05:00:00+00:00,4308.83,4328.69,4291.37,4315.32,23.234916,2017-08-17 05:59:59.999000+00:00,100304.823567,102,21.448071,92608.279728
2017-08-17 06:00:00+00:00,4330.29,4345.45,4309.37,4324.35,7.229691,2017-08-17 06:59:59.999000+00:00,31282.312670,36,4.802861,20795.317224
2017-08-17 07:00:00+00:00,4316.62,4349.99,4287.41,4349.99,4.443249,2017-08-17 07:59:59.999000+00:00,19241.058300,25,2.602292,11291.347015
2017-08-17 08:00:00+00:00,4333.32,4377.85,4333.32,4360.69,0.972807,2017-08-17 08:59:59.999000+00:00,4239.503586,28,0.814655,3552.746817


In [11]:
raw.tail()

,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume
open_time,,,,,,,,,,
2026-09-30 19:00:00+00:00,83950.01,83958.01,83503.18,83603.16,502.36930,2026-09-30 19:59:59.999000+00:00,4.206134e+07,111486,214.13204,1.792962e+07
2026-09-30 20:00:00+00:00,83603.16,83771.16,83530.11,83673.99,420.08920,2026-09-30 20:59:59.999000+00:00,3.513784e+07,66420,203.24575,1.699947e+07
2026-09-30 21:00:00+00:00,83673.99,83818.83,83673.99,83757.72,302.08867,2026-09-30 21:59:59.999000+00:00,2.530123e+07,53750,147.36874,1.234217e+07
2026-09-30 22:00:00+00:00,83757.73,83845.00,83580.29,83685.85,214.89420,2026-09-30 22:59:59.999000+00:00,1.799614e+07,52982,98.85444,8.278697e+06
2026-09-30 23:00:00+00:00,83685.86,83836.07,83523.65,83623.60,691.27124,2026-09-30 23:59:59.999000+00:00,5.783167e+07,87918,355.48853,2.973726e+07


## Data dictionary

Binance returns 12 fields per candle. We keep 11; `ignore` is an unused field Binance always sets to 0.
Each row describes the candle that **opens** at the index timestamp and closes one hour later.

| Field | Type | Meaning |
|---|---|---|
| `open_time` (index) | datetime, UTC | Start of the hour |
| `open`, `high`, `low`, `close` | float, USDT | First, highest, lowest and last trade price in the hour |
| `volume` | float, BTC | Base-asset volume traded |
| `close_time` | datetime, UTC | End of the hour (`open_time + 1h − 1ms`) |
| `quote_asset_volume` | float, USDT | Volume in USDT (≈ volume × average price) |
| `number_of_trades` | int | Number of individual trades |
| `taker_buy_base_volume` | float, BTC | Volume where the **buyer** was the aggressor (market buy) |
| `taker_buy_quote_volume` | float, USDT | Same, in USDT |

`taker_buy_base_volume / volume` measures **buy pressure**: 0.5 means balanced, above 0.5 means buyers are crossing the spread.
This is extra information that a plain OHLCV dataset would not have.

In [12]:
raw.describe().T.style.format("{:,.2f}")

,count,mean,std,min,25%,50%,75%,max
open,"79,836.00","39,000.80","32,501.30","2,870.90","9,442.01","29,144.61","63,097.68","126,011.18"
high,"79,836.00","39,154.50","32,596.39","2,950.00","9,487.65","29,214.51","63,294.15","126,199.63"
low,"79,836.00","38,839.81","32,403.14","2,817.00","9,400.00","29,056.68","62,875.03","125,252.74"
close,"79,836.00","39,001.79","32,501.46","2,919.00","9,442.94","29,145.03","63,098.44","126,011.18"
volume,"79,836.00","2,457.43","3,797.96",0.00,691.10,"1,311.98","2,576.48","137,207.19"
quote_asset_volume,"79,836.00","72,202,266.22","100,385,284.09",0.00,"14,456,864.84","39,228,176.27","91,048,468.33","3,005,633,669.26"
number_of_trades,"79,836.00","84,254.22","112,464.59",0.00,"17,447.25","44,679.50","103,031.25","3,520,247.00"
taker_buy_base_volume,"79,836.00","1,220.23","1,891.37",0.00,339.34,657.94,"1,282.52","68,611.45"
taker_buy_quote_volume,"79,836.00","35,645,878.36","49,992,433.45",0.00,"7,087,620.09","18,932,413.16","44,850,696.49","1,502,976,679.73"


## First observations

* **Scale changes enormously.** The price ranges from a few thousand to over one hundred thousand USDT. Any analysis on raw prices will be dominated by the recent years, which is why the EDA works on **log returns**.
* **Volume in BTC is not comparable across years.** The same BTC volume represents vastly different USDT amounts in 2017 and 2026. `quote_asset_volume` (USDT) is closer to "money traded", but it grows with the price too.
* All columns have the expected types, and there are no nulls in the raw file. Missing *hours* are invisible here because Binance simply omits those rows. **Notebook 02** reveals them by comparing against a complete hourly grid.